# 180 — Assemble the Zenodo release

This notebook converts the manuscript's notebook-by-notebook Zenodo checklist into a reproducible release bundle. It:

1. discovers the repository without relying on the launch directory;
2. collects canonical code, minimal observational inputs, compact derived tables, publication figures, and regional waveform/response records;
3. excludes regenerable Pickles, NPZ search surfaces, per-event plot galleries, PNG frame caches, and duplicate audio by default;
4. records every included file's size and SHA-256 digest;
5. writes machine-readable and human-readable manifests; and
6. reports unresolved scientific evidence and permission-dependent material separately from missing runtime files.

The assembler never edits source files. It builds through a temporary directory and refuses to replace an existing release unless explicitly configured. A release can be assembled while open evidence items remain, but its status will be **INCOMPLETE** until the required file and author-action reports are resolved.


**Saved execution note.** The outputs below are from a validated audit-only run (`ZENODO_BUILD=0`) on 15 September 2026. The default configuration builds a release; rerun all cells to assemble it.


In [1]:
from __future__ import annotations

import csv
import dataclasses
import hashlib
import json
import os
import shutil
import sys
import tempfile
import zipfile
from datetime import datetime, timezone
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "modules" / "project_config.py").is_file():
            return candidate
    raise FileNotFoundError("Could not locate modules/project_config.py")


PROJECT_ROOT = find_project_root(Path.cwd())
RELEASE_NAME = os.environ.get("ZENODO_RELEASE_NAME", "falcon9_seismoacoustic_v46")
RELEASE_PARENT = Path(
    os.environ.get("ZENODO_RELEASE_PARENT", PROJECT_ROOT / "release")
).expanduser().resolve()
RELEASE_ROOT = RELEASE_PARENT / RELEASE_NAME
ZIP_FILE = RELEASE_PARENT / f"{RELEASE_NAME}.zip"

# Optional directory containing main46.tex, supplement44.tex, the current
# scientific audit, and the current checklist if they have not yet been copied
# into the repository.
_review_assets = os.environ.get("ZENODO_REVIEW_ASSETS")
REVIEW_ASSETS_ROOT = Path(_review_assets).expanduser().resolve() if _review_assets else None

BUILD_RELEASE = os.environ.get("ZENODO_BUILD", "1") == "1"
REPLACE_EXISTING = os.environ.get("ZENODO_REPLACE", "0") == "1"
STRICT_REQUIRED_FILES = os.environ.get("ZENODO_STRICT", "0") == "1"
INCLUDE_CONVENIENCE_PDFS = os.environ.get("ZENODO_INCLUDE_PDFS", "1") == "1"
INCLUDE_REGIONAL_OBSERVATIONS = os.environ.get("ZENODO_INCLUDE_REGIONAL", "1") == "1"
INCLUDE_PERMISSION_DEPENDENT_VIDEO = os.environ.get("ZENODO_INCLUDE_VIDEO", "0") == "1"
MAKE_ZIP = os.environ.get("ZENODO_MAKE_ZIP", "1") == "1"

print("Project:", PROJECT_ROOT)
print("Release:", RELEASE_ROOT)
print("Build enabled:", BUILD_RELEASE)
print("Review assets:", REVIEW_ASSETS_ROOT or "not configured")


Project: /Users/thompsong/Developer/KSCRocketSeismology/08_fireball_paper/falcon9-seismoacoustic-workflow-1.0.0
Release: /Users/thompsong/Developer/KSCRocketSeismology/08_fireball_paper/falcon9-seismoacoustic-workflow-1.0.0/release/falcon9_seismoacoustic_v46
Build enabled: True
Review assets: not configured


## Declarative file inventory

Each rule identifies either one exact file, a set of matching files, or the first available alternative. Destinations preserve repository-relative paths unless a rule explicitly places review material under `documentation/`. Adding a new scientific product should normally mean adding a rule here, rather than adding ad hoc copy commands.


In [2]:
@dataclasses.dataclass(frozen=True)
class Rule:
    label: str
    category: str
    patterns: tuple[str, ...]
    required: bool = True
    include: bool = True
    first_alternative: bool = False
    source_root: str = "project"  # project or review
    destination_prefix: str = ""
    note: str = ""


rules: list[Rule] = []
def add(label, category, *patterns, **kwargs):
    rules.append(Rule(label, category, tuple(patterns), **kwargs))

# Workflow source and environment.
add("Repository README", "code", "README.md")
add("Conda environment", "code", "environment.yml")
add("Software license", "code", "LICENSE", "LICENSE.*")
add("Citation metadata", "code", "CITATION.cff")
add("Python modules", "code", "modules/*.py")
for number, name in (
    ("000", "correct_bchh_instrument_response"),
    ("010", "prepare_analysis_inputs"),
    ("020", "analyze_ksc_weather"),
    ("030", "reconcile_manual_event_catalogues"),
    ("040", "validate_event_catalogue_with_array_processing"),
    ("050", "review_event_catalogue_waveforms"),
    ("060", "validate_infrasound_baseline_correction"),
    ("070", "prepare_event_waveform_products"),
    ("080", "analyze_planar_array_propagation"),
    ("090", "analyze_finite_distance_propagation"),
    ("100", "measure_event_amplitudes_and_acoustic_seismic_coupling"),
    ("110", "measure_named_events"),
    ("124", "estimate_acoustic_energetics"),
    ("130", "analyze_seismic_polarization"),
    ("140", "search_for_direct_seismic_waves"),
    ("141", "generate_supplement_event_table"),
    ("145", "analyze_regional_detectability_with_regional_validation"),
    ("150", "align_uslaunchreport_video"),
    ("160", "generate_synchronized_phase1_video"),
    ("165", "generate_video_waveform_figure"),
    ("170", "generate_publication_figures_fixed"),
):
    add(f"Notebook {number}", "code", f"notebooks/{number}_{name}.ipynb")
add("Release assembler", "code", "notebooks/180_assemble_zenodo_release.ipynb")

# Minimal shared observational inputs and original metadata.
add("Raw six-channel accident waveform", "observations", "data/miniseed/01_bchh_raw_event_window.mseed")
add("Original BCHH response metadata", "metadata", "data/metadata/KSC.xml")
add("NASA/KSC weather workbook", "observations", "data/metadata/nasa_weather_tower_data.xls")
add("Launch-pad and camera coordinates", "metadata", "data/metadata/launchpads_cameras.kml")
add("Original CSS site/calibration metadata", "metadata", "data/metadata/sitedb.calibration", "data/metadata/sitedb.site", "data/metadata/sitedb.sitechan")
for filename in ("manual_arrival_picks.csv", "legacy_infrasound_event_catalogue.csv", "legacy_catalog_157_events.csv"):
    add(f"Legacy export {filename}", "observations", f"data/legacy_export/{filename}")
add("Legacy extraction provenance", "metadata", "data/legacy_export/legacy_export_summary.txt")
add(
    "Six-channel 48-hour precursor-screen record", "observations",
    "data/miniseed/*48*h*.mseed", "data/miniseed/*48hour*.mseed",
    "data/miniseed/*precursor*.mseed", first_alternative=True,
    note="Must span the stated 37 h before and 11 h after the first signal."
)
add(
    "Calibration provenance note", "documentation",
    "documentation/calibration_provenance.md", "docs/calibration_provenance.md",
    first_alternative=True,
)
add(
    "Manual video/event timing picks", "timing",
    "data/outputs/150_align_uslaunchreport_video/**/*.json",
    note="Consequential manual picks and revision/provenance record."
)
add(
    "Bracketing-launch calibration comparison", "open_evidence",
    "data/calibration_checks/**/*", required=False,
    note="Open scientific evidence task; absence does not prevent the assembler from running."
)
add(
    "Historical AGU slide/report evidence", "open_evidence",
    "documentation/*AGU*", "docs/*AGU*", required=False,
    note="Needed only if a specific historical no-shock attribution is restored."
)

# Compact outputs. CSV/JSON/XML/MiniSEED are retained; large regenerable object
# caches and event-by-event galleries are deliberately not selected.
output_dirs = (
    "000_correct_bchh_instrument_response",
    "010_prepare_analysis_inputs",
    "020_analyze_ksc_weather",
    "030_reconcile_manual_event_catalogues",
    "040_validate_event_catalogue_with_array_processing",
    "050_review_event_catalogue_waveforms",
    "060_validate_infrasound_baseline_correction",
    "070_prepare_event_waveform_products",
    "080_analyze_planar_array_propagation",
    "090_analyze_finite_distance_propagation",
    "100_measure_event_amplitudes_and_acoustic_seismic_coupling",
    "110_measure_named_events",
    "120_estimate_acoustic_energetics",  # legacy directory, producer is 124
    "130_analyze_seismic_polarization",
    "140_search_for_direct_seismic_waves",
    "145_analyze_regional_detectability",
    "150_align_uslaunchreport_video",
    "160_generate_synchronized_phase1_video",
    "165_generate_video_waveform_figure",
    "170_generate_publication_figures",
)
for directory in output_dirs:
    base = f"data/outputs/{directory}"
    add(f"{directory} root tables/provenance", "derived", f"{base}/*.csv", f"{base}/*.json", required=False)
    add(f"{directory} root portable metadata/waveforms", "derived", f"{base}/*.xml", f"{base}/*.mseed", required=False)

# Selected nested tables/provenance and publication figures; exclude named
# event_diagnostics, event_figures, surfaces, frame caches, and Pickles/NPZ.
for pattern in (
    "data/outputs/060_validate_infrasound_baseline_correction/derived/*.csv",
    "data/outputs/060_validate_infrasound_baseline_correction/derived/*.json",
    "data/outputs/070_prepare_event_waveform_products/derived/*.csv",
    "data/outputs/070_prepare_event_waveform_products/derived/*.json",
    "data/outputs/130_analyze_seismic_polarization/multiband/*.csv",
    "data/outputs/130_analyze_seismic_polarization/multiband/*.json",
):
    add("Nested compact result", "derived", pattern, required=False)

if INCLUDE_CONVENIENCE_PDFS:
    for directory in output_dirs:
        add(f"{directory} root PDFs", "figures", f"data/outputs/{directory}/*.pdf", required=False)
    for pattern in (
        "data/outputs/020_analyze_ksc_weather/figures/*.pdf",
        "data/outputs/030_reconcile_manual_event_catalogues/figures/*.pdf",
        "data/outputs/040_validate_event_catalogue_with_array_processing/figures/*.pdf",
        "data/outputs/060_validate_infrasound_baseline_correction/summary_figures/*.pdf",
        "data/outputs/080_analyze_planar_array_propagation/composites/*.pdf",
        "data/outputs/090_analyze_finite_distance_propagation/composites/*.pdf",
        "data/outputs/100_measure_event_amplitudes_and_acoustic_seismic_coupling/figures/*.pdf",
        "data/outputs/130_analyze_seismic_polarization/multiband/*.pdf",
        "data/outputs/140_search_for_direct_seismic_waves/figures/*.pdf",
    ):
        add("Selected publication/summary figures", "figures", pattern, required=False)

# Regional observations are the one intentional per-station collection.
if INCLUDE_REGIONAL_OBSERVATIONS:
    add("Regional MiniSEED records", "regional_observations", "data/outputs/145_analyze_regional_detectability/waveform_cache/**/*.mseed")
    add("Regional response StationXML", "regional_metadata", "data/outputs/145_analyze_regional_detectability/response_cache/**/*.xml")
    add("Regional event-epoch inventory", "regional_metadata", "data/outputs/145_analyze_regional_detectability/regional_station_inventory.xml")

# Manuscript/review assets may be installed in the repository or supplied by
# ZENODO_REVIEW_ASSETS. These alternatives are resolved below.
for filename in ("main46.tex", "supplement44.tex", "mybibfile40.bib"):
    add(filename, "manuscript", f"latex/{filename}", required=True)
for pattern in ("latex/*.cls", "latex/*.sty", "latex/*.bst", "latex/figures/*.pdf", "latex/generated/*.tex", "latex/generated/*.json"):
    add("LaTeX support", "manuscript", pattern, required=False)
if REVIEW_ASSETS_ROOT:
    for filename in ("main46.tex", "supplement44.tex", "mybibfile40.bib", "zenodo_checklist.md", "scientific_issue_resolution_audit_43.md"):
        add(filename, "documentation", filename, required=filename in {"main46.tex", "supplement44.tex", "mybibfile40.bib"}, source_root="review", destination_prefix="documentation/review_checkpoint")

# Permission-dependent movie is disabled unless explicitly authorized for the
# release profile. Configuration/provenance JSON and CSV are already included.
if INCLUDE_PERMISSION_DEPENDENT_VIDEO:
    add("Final synchronized movie", "permission_dependent", "data/outputs/160_generate_synchronized_phase1_video/*.mp4")

# Items that cannot be inferred honestly from filenames.
author_actions = [
    ("BLOCKER", "Supply the six-channel 48-hour precursor-screen MiniSEED and coverage/visual-review provenance."),
    ("BLOCKER", "Complete and freeze manual classification of the 93 regional automatic candidates, or preserve manuscript wording that states the review is incomplete."),
    ("BLOCKER", "Resolve/document the initial and principal physical pulse-to-PGV assignments used by Table 1."),
    ("OPEN_EVIDENCE", "Complete bracketing-launch calibration comparisons; document sensor-head/serial assignments and limits on over-range response."),
    ("OPEN_EVIDENCE", "Archive the exact AGU slides/report only if restoring the specific historical no-shock attribution."),
    ("RELEASE", "Add a calibration-provenance note distinguishing adopted empirical factors, nominal response, recollection, and unresolved cause."),
    ("RELEASE", "Record licenses/citations for data, Cartopy basemaps, InfraPy, TwistPy, and the source video."),
    ("PERMISSION", "Confirm whether the source/final USLaunchReport movie and derived audio may be redistributed before enabling video inclusion."),
    ("RELEASE", "Install the final version-46 TeX/support assets and perform a clean portable rerun and final page proof."),
    ("RELEASE", "Add Zenodo DOI/version metadata after reservation; do not invent a DOI."),
]
print(f"Declared {len(rules)} file rules and {len(author_actions)} author actions")


Declared 128 file rules and 10 author actions


## Resolve and audit the inventory

The resulting table distinguishes missing required files from optional products. Overlapping rules are deduplicated by destination. Any destination collision between different source files is a hard error.


In [3]:
def root_for(rule: Rule) -> Path | None:
    return PROJECT_ROOT if rule.source_root == "project" else REVIEW_ASSETS_ROOT


def resolve_rule(rule: Rule):
    root = root_for(rule)
    if root is None or not rule.include:
        return []
    groups = []
    for pattern in rule.patterns:
        matches = sorted(p for p in root.glob(pattern) if p.is_file())
        groups.append((pattern, matches))
    if rule.first_alternative:
        for pattern, matches in groups:
            if matches:
                return [(pattern, matches[0])]
        return []
    return [(pattern, path) for pattern, matches in groups for path in matches]


resolved_rows = []
selected: dict[str, tuple[Path, Rule]] = {}
missing_required = []
for rule in rules:
    matches = resolve_rule(rule)
    if not matches and rule.required and rule.include:
        missing_required.append(rule)
    resolved_rows.append({
        "label": rule.label,
        "category": rule.category,
        "required": rule.required,
        "match_count": len(matches),
        "status": "FOUND" if matches else ("MISSING" if rule.required else "OPTIONAL_ABSENT"),
        "patterns": "; ".join(rule.patterns),
        "note": rule.note,
    })
    root = root_for(rule)
    for _, source in matches:
        relative = source.relative_to(root)
        destination = Path(rule.destination_prefix) / relative
        key = destination.as_posix()
        previous = selected.get(key)
        if previous and previous[0].resolve() != source.resolve():
            raise RuntimeError(f"Destination collision: {key}: {previous[0]} versus {source}")
        selected[key] = (source, rule)

excluded_suffixes = {".pkl", ".pickle", ".npz"}
for destination, (source, _) in selected.items():
    if source.suffix.lower() in excluded_suffixes:
        raise RuntimeError(f"Regenerable object cache was unexpectedly selected: {source}")
    if "frames_phase1" in source.parts or "event_diagnostics" in source.parts or "event_figures" in source.parts or "surfaces" in source.parts:
        raise RuntimeError(f"Large diagnostic/cache directory was unexpectedly selected: {source}")

selected_bytes = sum(source.stat().st_size for source, _ in selected.values())
print(f"Selected {len(selected):,} unique files ({selected_bytes / 1e6:.1f} MB)")
print(f"Missing required rules: {len(missing_required)}")
for rule in missing_required:
    print("  -", rule.label, "=>", "; ".join(rule.patterns))


Selected 721 unique files (279.2 MB)
Missing required rules: 3
  - Six-channel 48-hour precursor-screen record => data/miniseed/*48*h*.mseed; data/miniseed/*48hour*.mseed; data/miniseed/*precursor*.mseed
  - Calibration provenance note => documentation/calibration_provenance.md; docs/calibration_provenance.md
  - Manual video/event timing picks => data/outputs/150_align_uslaunchreport_video/**/*.json


## Assemble, checksum, and archive

Set `ZENODO_BUILD=0` for audit-only operation. With building enabled, the notebook writes to a temporary sibling and atomically installs the completed staging directory. Set `ZENODO_REPLACE=1` only when intentionally replacing an earlier local staging build.


In [4]:
def sha256_file(path: Path, chunk_size=1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        while chunk := stream.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def write_reports(stage: Path, manifest_rows: list[dict]):
    documentation = stage / "documentation"
    documentation.mkdir(parents=True, exist_ok=True)
    with (documentation / "file_manifest.csv").open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(manifest_rows[0]) if manifest_rows else ["path"])
        writer.writeheader(); writer.writerows(manifest_rows)
    (documentation / "file_manifest.json").write_text(json.dumps(manifest_rows, indent=2) + "\n")
    (documentation / "inventory_rule_status.json").write_text(json.dumps(resolved_rows, indent=2) + "\n")
    actions = [{"status": status, "action": action} for status, action in author_actions]
    (documentation / "outstanding_author_actions.json").write_text(json.dumps(actions, indent=2) + "\n")
    status = "INCOMPLETE" if missing_required or any(s == "BLOCKER" for s, _ in author_actions) else "READY_FOR_FINAL_REVIEW"
    lines = [
        "# Falcon 9 seismo-acoustic Zenodo release staging report", "",
        f"Generated: {datetime.now(timezone.utc).isoformat()}",
        f"Status: **{status}**", "",
        f"Files: {len(manifest_rows):,}",
        f"Payload size: {sum(row['size_bytes'] for row in manifest_rows) / 1e6:.1f} MB", "",
        "## Missing required file rules", "",
    ]
    lines += [f"- {r.label}: `{'; '.join(r.patterns)}`" for r in missing_required] or ["- None"]
    lines += ["", "## Outstanding author actions", ""]
    lines += [f"- **{s}** — {a}" for s, a in author_actions]
    lines += ["", "## Deliberately excluded by the minimal profile", "",
              "- Pickle caches and NPZ search surfaces", "- Per-event PDF galleries and diagnostic plots",
              "- Rendered PNG movie-frame caches", "- Duplicate/generated audio unless redistribution is authorized",
              "- The source video unless separately licensed and explicitly enabled", ""]
    (documentation / "ASSEMBLY_STATUS.md").write_text("\n".join(lines))
    return status


if STRICT_REQUIRED_FILES and missing_required:
    raise FileNotFoundError("Required release files are missing; see the audit above")

if BUILD_RELEASE:
    RELEASE_PARENT.mkdir(parents=True, exist_ok=True)
    if RELEASE_ROOT.exists() and not REPLACE_EXISTING:
        raise FileExistsError(f"Release already exists: {RELEASE_ROOT}; set ZENODO_REPLACE=1 to replace it")
    temporary = Path(tempfile.mkdtemp(prefix=f".{RELEASE_NAME}-", dir=RELEASE_PARENT))
    manifest_rows = []
    try:
        for destination, (source, rule) in sorted(selected.items()):
            target = temporary / destination
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, target)
            manifest_rows.append({
                "path": destination,
                "category": rule.category,
                "size_bytes": target.stat().st_size,
                "sha256": sha256_file(target),
                "source_rule": rule.label,
            })
        release_status = write_reports(temporary, manifest_rows)
        if RELEASE_ROOT.exists():
            shutil.rmtree(RELEASE_ROOT)
        temporary.replace(RELEASE_ROOT)
    except Exception:
        shutil.rmtree(temporary, ignore_errors=True)
        raise

    if MAKE_ZIP:
        temporary_zip = ZIP_FILE.with_suffix(".zip.tmp")
        if temporary_zip.exists(): temporary_zip.unlink()
        with zipfile.ZipFile(temporary_zip, "w", compression=zipfile.ZIP_DEFLATED, allowZip64=True) as archive:
            for path in sorted(RELEASE_ROOT.rglob("*")):
                if path.is_file(): archive.write(path, Path(RELEASE_NAME) / path.relative_to(RELEASE_ROOT))
        temporary_zip.replace(ZIP_FILE)
        print("ZIP:", ZIP_FILE, f"({ZIP_FILE.stat().st_size / 1e6:.1f} MB)")
    print("Staging directory:", RELEASE_ROOT)
    print("Release status:", release_status)
else:
    print("Audit only; no release directory was written.")


ZIP: /Users/thompsong/Developer/KSCRocketSeismology/08_fireball_paper/falcon9-seismoacoustic-workflow-1.0.0/release/falcon9_seismoacoustic_v46.zip (208.6 MB)
Staging directory: /Users/thompsong/Developer/KSCRocketSeismology/08_fireball_paper/falcon9-seismoacoustic-workflow-1.0.0/release/falcon9_seismoacoustic_v46
Release status: INCOMPLETE


## Final verification

This verifies manifest hashes and confirms that no symlinks escaped into the package. It does not claim that open scientific evidence, permissions, DOI metadata, or a clean-environment rerun has been completed.


In [5]:
if BUILD_RELEASE:
    manifest = json.loads((RELEASE_ROOT / "documentation" / "file_manifest.json").read_text())
    failures = []
    for row in manifest:
        path = RELEASE_ROOT / row["path"]
        if not path.is_file() or path.is_symlink():
            failures.append((row["path"], "missing or symlink"))
        elif path.stat().st_size != row["size_bytes"]:
            failures.append((row["path"], "size mismatch"))
        elif sha256_file(path) != row["sha256"]:
            failures.append((row["path"], "SHA-256 mismatch"))
    if failures:
        raise RuntimeError(f"Manifest verification failed: {failures[:10]}")
    print(f"Verified all {len(manifest):,} manifested files")
    print((RELEASE_ROOT / "documentation" / "ASSEMBLY_STATUS.md").read_text())


Verified all 721 manifested files
# Falcon 9 seismo-acoustic Zenodo release staging report

Generated: 2026-09-15T02:30:29.553963+00:00
Status: **INCOMPLETE**

Files: 721
Payload size: 279.3 MB

## Missing required file rules

- Six-channel 48-hour precursor-screen record: `data/miniseed/*48*h*.mseed; data/miniseed/*48hour*.mseed; data/miniseed/*precursor*.mseed`
- Calibration provenance note: `documentation/calibration_provenance.md; docs/calibration_provenance.md`
- Manual video/event timing picks: `data/outputs/150_align_uslaunchreport_video/**/*.json`

## Outstanding author actions

- **BLOCKER** — Supply the six-channel 48-hour precursor-screen MiniSEED and coverage/visual-review provenance.
- **BLOCKER** — Complete and freeze manual classification of the 93 regional automatic candidates, or preserve manuscript wording that states the review is incomplete.
- **BLOCKER** — Resolve/document the initial and principal physical pulse-to-PGV assignments used by Table 1.
- **OPEN_EVIDENC